# MultiVolumeViewer: several 3D fields, conditions and facets

`MultiVolumeViewer` (in `cdiutils.interactive`) is a Jupyter viewer for several volumes on the same grid (amplitude, phase, displacement, strain, ...), typically BCDI reconstructions. This notebook runs on **synthetic data**, so it needs no beamline files. It shows:

1. opening several conditions (e.g. pressure steps) with units,
2. building a view by script (surface strain, slice, colour ranges),
3. facet patches and statistics,
4. exports (PNG, GIF / MP4, interactive HTML) with the built-in renderer,
5. saving the view as a reusable script,
6. opening real cdiutils results (`from_cdiutils`, `launcher`).

Everything done here by script can also be done in the panel (tabs *Layers*, *Add*, *Analyze*, *Export*, *Script*, *Help*).

In [ ]:
from pathlib import Path

import numpy as np

from cdiutils.interactive import MultiVolumeViewer

OUT = Path("multivolume_viewer_outputs")
OUT.mkdir(exist_ok=True)

## 1. Synthetic particle

A faceted particle (cuboctahedron: {100} and {111} facets) with a strain field that grows over three "pressure" conditions. Arrays are indexed `(z, y, x)`; values outside the particle are NaN, as in cdiutils results.

In [ ]:
def particle(n=64, strain_scale=1.0):
    z, y, x = np.mgrid[:n, :n, :n].astype(float) - n / 2 + 0.3
    cube = np.maximum(np.maximum(abs(x), abs(y)), abs(z)) / 18
    octa = (abs(x) + abs(y) + abs(z)) / 27
    amp = np.clip(1.5 - np.maximum(cube, octa), 0, 1)
    inside = amp > 0.3
    r2 = (x**2 + y**2 + z**2) / n**2
    strain = strain_scale * (0.05 * x / n - 0.08 * r2 + 0.01)  # in %
    phase = 0.4 * strain_scale * np.sin(x / 9) * np.cos(y / 11)
    nan = lambda a: np.where(inside, a, np.nan).astype(np.float32)
    return {
        "amplitude": amp.astype(np.float32),
        "phase": nan(phase),
        "het_strain": nan(strain),
    }


data = [particle(strain_scale=s) for s in (1.0, 1.6, 2.4)]
print({k: v.shape for k, v in data[0].items()})

## 2. Open the viewer

A list of dicts gives one condition per entry (slider in the toolbar). `field_units` labels the colorbars; `het_strain` is already in % in cdiutils. `facet_families` and `orientation_matrix` (crystal frame) are used by the facet tools.

In [ ]:
v = MultiVolumeViewer(
    data,
    voxel_size=(5, 5, 5),
    unit="nm",
    condition_labels=["0 GPa", "1 GPa", "2 GPa"],
    field_units={"het_strain": "%"},     # phase gets "rad" by default
    facet_families="100, 111",
    figure_preset="paper",               # export size 1200 x 900
)
v

## 3. A view by script

Each step is an action with its main argument; the script is checked before anything runs. `range: sym99` is a colour range symmetric around zero from the 99th percentile of |strain|; `range_all` uses the same range in every condition.

In [ ]:
v.run_actions([
    {"edit": "amplitude", "thr": 0.5, "color_by": "het_strain", "cmap": "RdBu_r",
     "range": "sym99", "range_all": True},
    {"slice": "z-slice", "source": "het_strain", "axis": "z", "cmap": "RdBu_r",
     "range": "sym99", "range_all": True, "show_colorbar": False},
    {"camera": "iso"},
])

In [ ]:
# a mistake is reported with a suggestion, and nothing runs
try:
    v.run_actions([{"edit": "amplitude", "cmpa": "viridis"}])
except Exception as e:
    print(e)

## 4. Facets

Faces of the isosurface are assigned to the closest crystal plane; edge-connected faces of the same plane form a facet patch. The table gives the area, the deviation from the plane and the strain on each facet.

In [ ]:
import pandas as pd

v.run_actions([{"hide": "z-slice"}])
rows = v.facet_patches("amplitude", "het_strain", labels=True, colour=True)
v.save_facet_csv(OUT / "facets.csv", "amplitude", "het_strain")
pd.DataFrame(rows)[["plane", "family", "area_frac", "deviation_deg", "mean", "std"]].round(4)

In [ ]:
# surface shell (10 nm) vs core statistics of the strain
st = v.region_stats("het_strain", "amplitude", threshold=0.5, depth=10)
pd.DataFrame({k: st[k] for k in ("shell", "core", "all")}).T.round(4)

## 5. Exports

Images and animations use the built-in renderer (numpy + matplotlib): no Kaleido, browser or GPU. GIF and MP4 need no system ffmpeg. In a notebook, `await` the animations to chain them.

In [ ]:
v.run_actions([{"edit": "amplitude", "color_by": "het_strain"}, {"hide": "amplitude:facets"}])
v.write_image(OUT / "surface_strain.png")
await v.start_rotation_export("orbit_z", frames=36, fmt="gif", name=str(OUT / "rotation"))
await v.start_condition_export(hold=10, fmt="gif", name=str(OUT / "conditions"),
                               label_format="{label}")
v.export_html(OUT / "view.html", conditions=True)   # slider over the conditions

In [ ]:
from IPython.display import Image

Image(filename=str(OUT / "conditions.gif"), width=500)

## 6. Keep the view as a script

`save_actions` writes the current view (layers, styles, units, camera) as YAML; `run_actions` replays it, here on a new viewer. The *Script* tab does the same in the panel.

In [ ]:
path = v.save_actions(OUT / "view.yaml")
print(path.read_text())
w = MultiVolumeViewer(data, voxel_size=(5, 5, 5), unit="nm",
                      condition_labels=["0 GPa", "1 GPa", "2 GPa"])
w.run_actions(path)
w

## 7. Opening cdiutils results

`from_cdiutils` reads `S*_postprocessed_data.cxi`, `S*_structural_properties.npz` or `.vti` files (a file, a scan folder or a folder of scans), with the voxel size, isosurface and units of the pipeline, and prints the configuration first. Several scans become conditions, or layers side by side (`scans_as="layers"`). `launcher` shows a form to pick them. Here a small results folder is written first, as the pipeline would.

In [ ]:
d = OUT / "results" / "Sample" / "S12"
d.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    d / "S12_structural_properties.npz",
    **{k: np.nan_to_num(a) for k, a in data[0].items()},
    support=(data[0]["amplitude"] > 0.3).astype(float),
    voxel_size=np.array([5.0, 5.0, 5.0]),
)
r = MultiVolumeViewer.from_cdiutils(OUT / "results", label_from="sample")
r

In [ ]:
MultiVolumeViewer.launcher(OUT / "results")

## 8. Help

`v.help()` lists the topics (actions, colours, conditions, facets, export, ...), `v.help("clip")` explains one action with examples using your layer names, and `v.tutorial()` is a guided tour on your own data.

In [ ]:
v.help("colours")

In [ ]:
for viewer in (v, w, r):
    viewer.close()